# Large Model Probing — Pythia-6.9B, Pythia-12B, Mistral-7B-v0.1

**Phase 2:** Behavioral baseline and race/ethnicity probing for large models.

**Hardware:** A100 80GB  
**Runtime:** ~60 minutes

**Run all cells top to bottom.**

**Expected output:** 99-100% probe accuracy at all layers for all models.


In [7]:
# Configuration
DRIVE_BASE = '/content/drive/MyDrive/ID-UTH-repo'  # change if needed

import os
from google.colab import drive
try:
    drive.mount('/content/drive')
except ValueError:
    pass  # already mounted

os.makedirs(f'{DRIVE_BASE}/activations', exist_ok=True)
print(f'Drive base: {DRIVE_BASE}')
print('Ready.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive base: /content/drive/MyDrive/ID-UTH-repo
Ready.


In [8]:
# Setup
!pip install -q numpy==1.26.4
!pip install -q transformer_lens datasets scikit-learn

import torch, pickle, json, os
import numpy as np
from datasets import load_dataset
from transformer_lens import HookedTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

DEVICE = 'cuda'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Ready.')


GPU: NVIDIA A100-SXM4-40GB
Ready.


In [9]:
# Helper functions

def load_bbq_probing(split_name, targets, n_per_class=50):
    """Load BBQ from HuggingFace and return balanced records for probing."""
    dataset = load_dataset('Elfsong/BBQ', split=split_name)
    records = []
    for item in dataset:
        ctx = item['context']
        for t in targets:
            if f' {t} ' in ctx or ctx.lower().startswith(t.lower()):
                records.append({'text': ctx, 'label': t})
                break
    balanced = []
    for t in targets:
        subset = [r for r in records if r['label'] == t][:n_per_class]
        balanced.extend(subset)
    print(f'  Loaded {len(balanced)} records ({len(balanced)//len(targets)} per class)')
    return balanced

def extract_and_probe(model, records, layers):
    """Extract residual stream activations at final token and run logistic regression probe."""
    all_acts = {l: [] for l in layers}
    labels = []
    for i, rec in enumerate(records):
        tokens = model.to_tokens(rec['text'])
        with torch.no_grad():
            _, cache = model.run_with_cache(tokens)
        for l in layers:
            vec = cache[f'blocks.{l}.hook_resid_post'][0, -1, :].float().cpu().numpy()
            all_acts[l].append(vec)
        labels.append(rec['label'])
        if (i + 1) % 20 == 0:
            print(f'    Extracted {i+1}/{len(records)}')

    le = LabelEncoder()
    y = le.fit_transform(labels)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    results = {}
    for l in layers:
        X = np.array(all_acts[l])
        scores = []
        for train_idx, test_idx in cv.split(X, y):
            clf = LogisticRegression(max_iter=1000, random_state=42)
            clf.fit(X[train_idx], y[train_idx])
            scores.append(clf.score(X[test_idx], y[test_idx]))
        results[l] = (round(np.mean(scores), 4), round(np.std(scores), 4))
        print(f'    Layer {l}: {np.mean(scores):.1%} ± {np.std(scores):.1%}')
    return results

print('Helpers defined.')


Helpers defined.


In [10]:
# Models to probe — run sequentially
# Each loads the model, runs probing, saves pkl, then frees memory

MODELS = [
    {
        'model_id': 'EleutherAI/pythia-6.9b',
        'name': 'Pythia-6.9B',
        'layers': [8, 16, 24, 31],  # Pythia-6.9B has 32 layers (0-31)
        'dtype': 'fp16',
    },
    {
        'model_id': 'EleutherAI/pythia-12b',
        'name': 'Pythia-12B',
        'layers': [9, 18, 27, 35],  # Pythia-12B has 36 layers (0-35)
        'dtype': 'fp16',
    },
    {
        'model_id': 'mistralai/Mistral-7B-v0.1',
        'name': 'Mistral-7B-v0.1',
        'layers': [6, 12, 18, 24, 31],
        'dtype': 'fp16',
    },
]

CATEGORIES = {
    'race_ethnicity': {
        'split': 'race_ethnicity',
        'targets': ['Hispanic', 'Black'],
    },
    'gender_identity': {
        'split': 'gender_identity',
        'targets': ['man', 'woman'],
    },
}

all_results = {}

for model_cfg in MODELS:
    model_name = model_cfg['name']
    print(f'\n{"="*60}')
    print(f'Model: {model_name}')
    print(f'{"="*60}')

    dtype = torch.float16 if model_cfg['dtype'] == 'fp16' else torch.float32
    model = HookedTransformer.from_pretrained(
        model_cfg['model_id'], device=DEVICE, dtype=dtype)
    model.eval()
    print('Loaded.')

    model_results = {}
    for cat_name, cat_cfg in CATEGORIES.items():
        print(f'\n  Category: {cat_name}')
        records = load_bbq_probing(cat_cfg['split'], cat_cfg['targets'])
        model_results[cat_name] = extract_and_probe(model, records, model_cfg['layers'])

    all_results[model_name] = model_results

    # Save
    save_path = f"{DRIVE_BASE}/activations/{model_name.lower().replace('-','_').replace('.','')}_results.pkl"
    with open(save_path, 'wb') as f:
        pickle.dump({'model': model_name, 'results': model_results}, f)
    print(f'  Saved: {save_path}')

    del model
    torch.cuda.empty_cache()
    print(f'  {model_name} done.')

print('\n\nALL MODELS COMPLETE')
print('='*60)
for model_name, res in all_results.items():
    print(f'\n{model_name}:')
    for cat, layer_res in res.items():
        print(f'  {cat}:')
        for layer, (mean, std) in layer_res.items():
            print(f'    Layer {layer}: {mean:.1%} ± {std:.1%}')
print('\nExpected: 99-100% at all layers for all models.')



Model: Pythia-6.9B


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded pretrained model EleutherAI/pythia-6.9b into HookedTransformer
Loaded.

  Category: race_ethnicity
  Loaded 100 records (50 per class)
    Extracted 20/100
    Extracted 40/100
    Extracted 60/100
    Extracted 80/100
    Extracted 100/100
    Layer 8: 100.0% ± 0.0%
    Layer 16: 97.0% ± 4.0%
    Layer 24: 98.0% ± 2.4%
    Layer 31: 98.0% ± 2.4%

  Category: gender_identity
  Loaded 100 records (50 per class)
    Extracted 20/100
    Extracted 40/100
    Extracted 60/100
    Extracted 80/100
    Extracted 100/100
    Layer 8: 99.0% ± 2.0%
    Layer 16: 100.0% ± 0.0%
    Layer 24: 100.0% ± 0.0%
    Layer 31: 100.0% ± 0.0%
  Saved: /content/drive/MyDrive/ID-UTH-repo/activations/pythia_69b_results.pkl
  Pythia-6.9B done.

Model: Pythia-12B


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model EleutherAI/pythia-12b into HookedTransformer
Loaded.

  Category: race_ethnicity
  Loaded 100 records (50 per class)
    Extracted 20/100
    Extracted 40/100
    Extracted 60/100
    Extracted 80/100
    Extracted 100/100
    Layer 9: 100.0% ± 0.0%
    Layer 18: 98.0% ± 2.4%
    Layer 27: 98.0% ± 2.4%
    Layer 35: 98.0% ± 2.4%

  Category: gender_identity
  Loaded 100 records (50 per class)
    Extracted 20/100
    Extracted 40/100
    Extracted 60/100
    Extracted 80/100
    Extracted 100/100
    Layer 9: 99.0% ± 2.0%
    Layer 18: 100.0% ± 0.0%
    Layer 27: 100.0% ± 0.0%
    Layer 35: 100.0% ± 0.0%
  Saved: /content/drive/MyDrive/ID-UTH-repo/activations/pythia_12b_results.pkl
  Pythia-12B done.

Model: Mistral-7B-v0.1


...

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Loaded pretrained model mistralai/Mistral-7B-v0.1 into HookedTransformer
Loaded.

  Category: race_ethnicity
  Loaded 100 records (50 per class)
    Extracted 20/100
    Extracted 40/100
    Extracted 60/100
    Extracted 80/100
    Extracted 100/100
    Layer 6: 68.0% ± 16.9%
    Layer 12: 96.0% ± 3.7%
    Layer 18: 99.0% ± 2.0%
    Layer 24: 100.0% ± 0.0%
    Layer 31: 100.0% ± 0.0%

  Category: gender_identity
  Loaded 100 records (50 per class)
    Extracted 20/100
    Extracted 40/100
    Extracted 60/100
    Extracted 80/100
    Extracted 100/100
    Layer 6: 99.0% ± 2.0%
    Layer 12: 100.0% ± 0.0%
    Layer 18: 100.0% ± 0.0%
    Layer 24: 100.0% ± 0.0%
    Layer 31: 100.0% ± 0.0%
  Saved: /content/drive/MyDrive/ID-UTH-repo/activations/mistral_7b_v01_results.pkl
  Mistral-7B-v0.1 done.


ALL MODELS COMPLETE

Pythia-6.9B:
  race_ethnicity:
    Layer 8: 100.0% ± 0.0%
    Layer 16: 97.0% ± 4.0%
    Layer 24: 98.0% ± 2.5%
    Layer 31: 98.0% ± 2.5%
  gender_identity:
    Layer 8: 99